# Security Agent PoC: AI‑driven Process Analysis with RAG

## Overview

This notebook demonstrates a modular AI agent that:

- Uses a **knowledge base** (OWASP cheat sheets) to provide security context.
- Scans running processes on the Linux system.
- Retrieves relevant knowledge chunks using **RAG** (Retrieval-Augmented Generation).
- Asks a local LLM (`gemma3:4b`) to evaluate each process for security risks.
- Produces a report with risk assessment and mitigation steps.

All components are separated into modules for clarity and reusability.


In [ ]:
from planner import SecurityPlanner
from rag import load_articles_to_chroma, retrieve_context
import json

In [ ]:
# Constant
MODEL = "gemma3:4b"

## Rebuild the Chroma collection with chunked documents

In [ ]:
load_articles_to_chroma()

In [ ]:
results = retrieve_context("Authorization", n_results=3)
print(results)

## Manually Analyse a Specific Process

In [ ]:
# Example: analyse the current shell (if you want to see what the model thinks)
import os
import sys
import psutil

planner = SecurityPlanner(model=MODEL)

shell = psutil.Process(os.getpid())
proc_info = {
    "pid": shell.pid,
    "name": shell.name(),
    "exe": shell.exe(),
    "cmdline": ' '.join(shell.cmdline()),
    "user": shell.username(),
    "parent": psutil.Process(shell.ppid()).name(),
    "connections": [],
    "open_files": []
}
analysis = planner.analyse_process(proc_info)
print(f"Analysis of current shell (PID {shell.pid}):")
print(json.dumps(analysis, indent=2))

## Run the planner

In [ ]:
planner = SecurityPlanner(model=MODEL)
results = planner.run(update_kb=True)

## Display Results

In [ ]:
for r in results:
    proc = r["process"]
    analysis = r["analysis"]
    print(f"Process: {proc['name']} (PID {proc['pid']})")
    print(f"  Command: {proc['cmdline']}")
    print(f"  User: {proc['user']}")
    print(f"  Network: {proc['connections']}")
    print(f"  Suspicious: {analysis.get('is_suspicious', '?')}")
    print(f"  Risk: {analysis.get('risk_level', '?')}")
    print(f"  Reasoning: {analysis.get('reasoning', '')[:200]}...")
    print(f"  Mitigation: {analysis.get('mitigation', '')[:200]}...")
    print("-" * 50)